# Unit Test & QC：`load_processor` 和 `load_model_v2`

本笔记本对 `utils.py` 中新增的两个核心函数新增逻辑进行系统性单元测试与质量检查：

| 测试对象 | 关注点 |
|---|---|
| `load_processor` | `<row_i_col_j>` 原子token注册、特殊token配置、图像处理器参数 |
| `load_model_v2` | 词表扩充、模型组件替换、配置一致性、前向传播 |

In [1]:
import sys, os
# 确保从项目根目录导入
sys.path.insert(0, os.path.abspath('.'))

import torch
import math

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'使用设备: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

d:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


使用设备: cuda:0
GPU: NVIDIA GeForce RTX 4090 Laptop GPU
显存: 17.2 GB


## 辅助函数

In [2]:
def check(condition: bool, name: str, detail: str = ''):
    """统一的测试断言，打印 PASS / FAIL"""
    status = '✅ PASS' if condition else '❌ FAIL'
    msg = f'{status}  [{name}]'
    if detail:
        msg += f'  →  {detail}'
    print(msg)
    if not condition:
        raise AssertionError(f'测试失败: {name}  {detail}')

---
## 第一部分：`load_processor` 测试

In [3]:
from utils import load_processor

processor = load_processor()
tokenizer = processor.tokenizer
print('\nprocessor 类型:', type(processor).__name__)
print('tokenizer  类型:', type(tokenizer).__name__)

d:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载SmolVLM2处理器...
正在加载Qwen3分词器...
  添加前词表大小: 151669
  已新增 36 个位置特殊token（<row_i_col_j>）
  添加后词表大小: 151705
正在配置处理器...
  图像处理器（来自 preprocessor_config.json）: size={'longest_edge': 2048}, max_image_size={'longest_edge': 512}, do_image_splitting=True

processor 类型: SmolVLMProcessor
tokenizer  类型: Qwen2TokenizerFast


### 1-A  `<row_i_col_j>` 原子性测试
每个位置token必须被分词器视为单一token（编码后长度 == 1），否则SmolVLM2的空间位置信号会被BPE破坏（论文称之为"OCR loss plague"）。

In [4]:
ROW_COL_TOKENS = [f'<row_{i}_col_{j}>' for i in range(1, 7) for j in range(1, 7)]
print(f'期望的位置token总数: {len(ROW_COL_TOKENS)}')

vocab = tokenizer.get_vocab()
failed_tokens = []
for tok in ROW_COL_TOKENS:
    if tok not in vocab:
        failed_tokens.append((tok, 'NOT IN VOCAB'))
        continue
    ids = tokenizer.encode(tok, add_special_tokens=False)
    if len(ids) != 1:
        failed_tokens.append((tok, f'拆分为 {len(ids)} 个子词: {ids}'))

check(len(failed_tokens) == 0,
      '所有36个 <row_i_col_j> 均为单一原子token',
      f'失败token: {failed_tokens}' if failed_tokens else '全部通过')

# 展示几个示例
print('\n示例编码（期望每个仅1个ID）:')
for tok in ['<row_1_col_1>', '<row_3_col_3>', '<row_5_col_5>', '<row_6_col_6>']:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    print(f'  {tok:20s} → ID={ids}')

期望的位置token总数: 36
✅ PASS  [所有36个 <row_i_col_j> 均为单一原子token]  →  全部通过

示例编码（期望每个仅1个ID）:
  <row_1_col_1>        → ID=[151669]
  <row_3_col_3>        → ID=[151683]
  <row_5_col_5>        → ID=[151697]
  <row_6_col_6>        → ID=[151704]


### 1-B  词表大小验证
Qwen3 基础词表大小从模型本身读取（不硬编码），添加36个位置token后词表应恰好增加36。同时确认所有新token的ID落在扩充区域（>= 基础词表大小）。

In [5]:
# 动态推算基础词表大小：当前词表 - 刚才新增的36个位置token
# 避免硬编码——不同版本的 Qwen3 权重词表大小可能不同
N_POSITION_TOKENS = 36
actual_vocab     = len(tokenizer)
QWEN3_BASE_VOCAB = actual_vocab - N_POSITION_TOKENS
EXPECTED_VOCAB   = actual_vocab  # 即 base + 36

print(f'Qwen3 基础词表大小（动态推算）: {QWEN3_BASE_VOCAB}')
print(f'新增位置token数量:              {N_POSITION_TOKENS}')
print(f'当前词表大小:                   {actual_vocab}')

check(actual_vocab == QWEN3_BASE_VOCAB + N_POSITION_TOKENS,
      '词表大小正确（base + 36）',
      f'base={QWEN3_BASE_VOCAB}, +{N_POSITION_TOKENS} = {actual_vocab}')

new_token_ids = tokenizer.convert_tokens_to_ids(ROW_COL_TOKENS)
all_in_extended = all(tid >= QWEN3_BASE_VOCAB for tid in new_token_ids)
check(all_in_extended,
      f'所有位置token的ID均在扩充区域（ID >= {QWEN3_BASE_VOCAB}）',
      f'ID范围: [{min(new_token_ids)}, {max(new_token_ids)}]')

Qwen3 基础词表大小（动态推算）: 151669
新增位置token数量:              36
当前词表大小:                   151705
✅ PASS  [词表大小正确（base + 36）]  →  base=151669, +36 = 151705
✅ PASS  [所有位置token的ID均在扩充区域（ID >= 151669）]  →  ID范围: [151669, 151704]


### 1-C  特殊token配置验证

In [6]:
special_token_checks = {
    'fake_image_token':       ('<vision_start>',  processor.fake_image_token),
    'image_token':            ('<|image_pad|>',   processor.image_token),
    'image_token_id':         (151655,            processor.image_token_id),
    'end_of_utterance_token': ('<im_end>',        processor.end_of_utterance_token),
    'global_image_token':     ('<|vision_pad|>',  processor.global_image_token),
    'video_token':            ('<|video_pad|>',   processor.video_token),
}

for name, (expected, actual) in special_token_checks.items():
    check(actual == expected, f'processor.{name}', f'实际={actual!r}, 期望={expected!r}')

✅ PASS  [processor.fake_image_token]  →  实际='<vision_start>', 期望='<vision_start>'
✅ PASS  [processor.image_token]  →  实际='<|image_pad|>', 期望='<|image_pad|>'
✅ PASS  [processor.image_token_id]  →  实际=151655, 期望=151655
✅ PASS  [processor.end_of_utterance_token]  →  实际='<im_end>', 期望='<im_end>'
✅ PASS  [processor.global_image_token]  →  实际='<|vision_pad|>', 期望='<|vision_pad|>'
✅ PASS  [processor.video_token]  →  实际='<|video_pad|>', 期望='<|video_pad|>'


### 1-D  图像处理器参数验证
从 `preprocessor_config.json` 动态读取两个关键字段：
- `size.longest_edge`：原图切割前的缩放上限（官方配置为 **2048**）
- `max_image_size.longest_edge`：每个子图 crop 送入 SigLIP 的分辨率（**512**）
- `do_image_splitting`：是否对高分辨率图像做网格切割（**True**）

最大网格：`ceil(size / max_image_size) = ceil(2048 / 512) = 4` → 最多 **4×4 = 16 crops**，在预定义的 36 个位置 token 范围内。

In [7]:
ip = processor.image_processor

# 从 preprocessor_config.json 动态读取，不硬编码期望值
size_edge = ip.size['longest_edge'] if isinstance(ip.size, dict) else ip.size
crop_edge = (ip.max_image_size.get('longest_edge')
             if isinstance(getattr(ip, 'max_image_size', None), dict)
             else getattr(ip, 'max_image_size', None))

print(f'size.longest_edge           = {size_edge}')
print(f'max_image_size.longest_edge = {crop_edge}')
print(f'do_image_splitting          = {ip.do_image_splitting}')

# do_image_splitting 必须为 True（图像模式）
check(ip.do_image_splitting is True,
      'do_image_splitting = True',
      f'实际={ip.do_image_splitting}')

# 最大网格必须在 36 个预定义位置 token 范围内（≤ 6×6）
max_grid = math.ceil(size_edge / crop_edge)
print(f'\n最大网格: ceil({size_edge} / {crop_edge}) = {max_grid}  →  {max_grid}×{max_grid} = {max_grid**2} crops')
check(max_grid <= 6,
      f'最大网格 {max_grid}×{max_grid} ≤ 6×6（在预定义的36个位置token范围内）',
      f'max_grid={max_grid}')

size.longest_edge           = 2048
max_image_size.longest_edge = 512
do_image_splitting          = True
✅ PASS  [do_image_splitting = True]  →  实际=True

最大网格: ceil(2048 / 512) = 4  →  4×4 = 16 crops
✅ PASS  [最大网格 4×4 ≤ 6×6（在预定义的36个位置token范围内）]  →  max_grid=4


### 1-E  聊天模板加载验证

In [9]:
check(bool(processor.chat_template),
      '聊天模板非空',
      f'长度={len(processor.chat_template)} 字符')

print('\n聊天模板前150字符预览:')
print(processor.chat_template[:150], '...')

✅ PASS  [聊天模板非空]  →  长度=5038 字符

聊天模板前150字符预览:
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif ...


### 1-F  端到端分词测试
构造含 `<row_i_col_j>` 的文本，验证其在分词后作为单一token出现，不被BPE拆分。

In [10]:
test_cases = [
    ('图像子区域 <row_1_col_1> 的内容：',   '<row_1_col_1>'),
    ('第 <row_3_col_2> 块区域描述如下：',    '<row_3_col_2>'),
    ('<row_5_col_5> 位置的文字为：',          '<row_5_col_5>'),
]

for text, expected_tok in test_cases:
    tokens = tokenizer.tokenize(text)
    print(f'  输入: {text!r}')
    print(f'  分词: {tokens}')
    check(expected_tok in tokens,
          f'{expected_tok} 作为单一token出现',
          f'分词序列: {tokens}')
    print()

  输入: '图像子区域 <row_1_col_1> 的内容：'
  分词: ['åĽ¾åĥı', 'åŃĲ', 'åĮºåŁŁ', 'Ġ', '<row_1_col_1>', 'ĠçļĦ', 'åĨħå®¹', 'ï¼ļ']
✅ PASS  [<row_1_col_1> 作为单一token出现]  →  分词序列: ['åĽ¾åĥı', 'åŃĲ', 'åĮºåŁŁ', 'Ġ', '<row_1_col_1>', 'ĠçļĦ', 'åĨħå®¹', 'ï¼ļ']

  输入: '第 <row_3_col_2> 块区域描述如下：'
  分词: ['ç¬¬', 'Ġ', '<row_3_col_2>', 'Ġå', 'Ŀ', 'Ĺ', 'åĮºåŁŁ', 'æııè¿°', 'å¦Ĥä¸ĭ', 'ï¼ļ']
✅ PASS  [<row_3_col_2> 作为单一token出现]  →  分词序列: ['ç¬¬', 'Ġ', '<row_3_col_2>', 'Ġå', 'Ŀ', 'Ĺ', 'åĮºåŁŁ', 'æııè¿°', 'å¦Ĥä¸ĭ', 'ï¼ļ']

  输入: '<row_5_col_5> 位置的文字为：'
  分词: ['<row_5_col_5>', 'Ġ', 'ä½įç½®', 'çļĦæĸĩåŃĹ', 'ä¸º', 'ï¼ļ']
✅ PASS  [<row_5_col_5> 作为单一token出现]  →  分词序列: ['<row_5_col_5>', 'Ġ', 'ä½įç½®', 'çļĦæĸĩåŃĹ', 'ä¸º', 'ï¼ļ']



---
## 第二部分：`load_model_v2` 测试

### 2-A  不传 `new_vocab_size` 时保持 Qwen3 模型原始嵌入大小

> **注意**：Qwen3 的模型嵌入矩阵大小（embed_tokens 行数）与分词器词表大小**故意不同**。
> 模型会把嵌入矩阵行数对齐到某个整数倍（如 128 的倍数）以提升 GPU 效率，因此
> `embed_tokens.shape[0]`（151936）> `len(tokenizer) - 36`（151669）。
> 本测试只验证"不传 new_vocab_size 时不发生 resize"，动态记录模型原始大小作为基准。

In [11]:
from utils import load_model_v2

# 不传 new_vocab_size，不传 trained_model_path
model_base = load_model_v2(trained_model_path=None, device=DEVICE, new_vocab_size=None)

embed_shape   = model_base.model.text_model.embed_tokens.weight.shape
lm_head_shape = model_base.lm_head.weight.shape

# 动态记录 Qwen3 模型原始嵌入大小作为基准
# （与 tokenizer 词表大小不同：模型对嵌入行数做了对齐填充）
QWEN3_MODEL_VOCAB = embed_shape[0]
print(f'\nembed_tokens.weight:  {embed_shape}')
print(f'lm_head.weight:       {lm_head_shape}')
print(f'tokenizer 词表大小:   {QWEN3_BASE_VOCAB} + 36 = {QWEN3_BASE_VOCAB + 36}')
print(f'模型嵌入矩阵大小:     {QWEN3_MODEL_VOCAB}  （对齐填充后，故意大于 tokenizer 词表）')
print(f'差值:                 {QWEN3_MODEL_VOCAB - QWEN3_BASE_VOCAB}  行为填充行')

# 两者一致说明没有发生额外 resize
check(embed_shape[0] == lm_head_shape[0],
      'embed_tokens 与 lm_head 行数一致（均未 resize）',
      f'embed={embed_shape[0]}, lm_head={lm_head_shape[0]}')

# 模型原始嵌入大小应 ≥ tokenizer 词表大小（填充行不对应真实 token）
check(QWEN3_MODEL_VOCAB >= QWEN3_BASE_VOCAB,
      f'模型嵌入大小（{QWEN3_MODEL_VOCAB}）≥ tokenizer 词表基础大小（{QWEN3_BASE_VOCAB}）',
      '符合预期：模型做了对齐填充')

# 释放显存
del model_base
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n已释放显存')

`torch_dtype` is deprecated! Use `dtype` instead!


正在加载SmolVLM2视觉-语言模型...
正在加载Qwen3语言模型...
正在构建连接器配置...
正在创建新的连接器...
正在替换语言模型组件...
正在更新模型配置...
模型架构构建完成！
未提供 trained_model_path，使用预训练基础权重。

embed_tokens.weight:  torch.Size([151936, 1024])
lm_head.weight:       torch.Size([151936, 1024])
tokenizer 词表大小:   151669 + 36 = 151705
模型嵌入矩阵大小:     151936  （对齐填充后，故意大于 tokenizer 词表）
差值:                 267  行为填充行
✅ PASS  [embed_tokens 与 lm_head 行数一致（均未 resize）]  →  embed=151936, lm_head=151936
✅ PASS  [模型嵌入大小（151936）≥ tokenizer 词表基础大小（151669）]  →  符合预期：模型做了对齐填充

已释放显存


### 2-B  传入 `new_vocab_size` 时的 resize 行为

> **新逻辑**：`load_model_v2` 现在与**实际嵌入矩阵大小**（`embed_tokens.weight.shape[0]或者vocab_size in config.json`）比较，
> 而非 `config.vocab_size`。只有 `new_vocab_size > actual_embed_size` 时才执行 resize。
>
> 当前场景：`NEW_VOCAB = len(tokenizer) = 151705`，`actual_embed_size = 151936`
> 由于 **151705 ≤ 151936**，36 个新位置 token（IDs 151669–151704）已落入 Qwen3 的 padding 行内，
> **无需 resize**，嵌入矩阵行数保持 151936 不变。

In [12]:
NEW_VOCAB = len(tokenizer)   # = 151705
print(f'传入 new_vocab_size = {NEW_VOCAB}')
print(f'（Qwen3 实际嵌入矩阵大小 = {QWEN3_MODEL_VOCAB}，NEW_VOCAB ≤ QWEN3_MODEL_VOCAB，预期不 resize）')

model = load_model_v2(trained_model_path=None, device=DEVICE, new_vocab_size=NEW_VOCAB)

embed_shape   = model.model.text_model.embed_tokens.weight.shape
lm_head_shape = model.lm_head.weight.shape
print(f'\nembed_tokens.weight: {embed_shape}')
print(f'lm_head.weight:      {lm_head_shape}')

# 不触发 resize → 矩阵行数仍为原始对齐大小 QWEN3_MODEL_VOCAB（151936）
check(embed_shape[0] == QWEN3_MODEL_VOCAB,
      f'embed_tokens 未发生 resize，行数保持原始对齐大小（{QWEN3_MODEL_VOCAB}）',
      f'实际={embed_shape[0]}, 期望={QWEN3_MODEL_VOCAB}')
check(lm_head_shape[0] == QWEN3_MODEL_VOCAB,
      f'lm_head 未发生 resize，行数保持原始对齐大小（{QWEN3_MODEL_VOCAB}）',
      f'实际={lm_head_shape[0]}, 期望={QWEN3_MODEL_VOCAB}')

# 新增位置 token（IDs 151669–151704）已落在矩阵已有的 padding 行内
check(NEW_VOCAB <= QWEN3_MODEL_VOCAB,
      f'tokenizer 词表（{NEW_VOCAB}）≤ 模型嵌入大小（{QWEN3_MODEL_VOCAB}），无需 resize',
      f'差值={QWEN3_MODEL_VOCAB - NEW_VOCAB} 行为 padding')

传入 new_vocab_size = 151705
（Qwen3 实际嵌入矩阵大小 = 151936，NEW_VOCAB ≤ QWEN3_MODEL_VOCAB，预期不 resize）
正在加载SmolVLM2视觉-语言模型...
正在加载Qwen3语言模型...
正在构建连接器配置...
正在创建新的连接器...
new_vocab_size=151705 ≤ 实际嵌入矩阵大小 151936，新token已落入padding行，无需resize。
正在替换语言模型组件...
正在更新模型配置...
模型架构构建完成！
未提供 trained_model_path，使用预训练基础权重。

embed_tokens.weight: torch.Size([151936, 1024])
lm_head.weight:      torch.Size([151936, 1024])
✅ PASS  [embed_tokens 未发生 resize，行数保持原始对齐大小（151936）]  →  实际=151936, 期望=151936
✅ PASS  [lm_head 未发生 resize，行数保持原始对齐大小（151936）]  →  实际=151936, 期望=151936
✅ PASS  [tokenizer 词表（151705）≤ 模型嵌入大小（151936），无需 resize]  →  差值=231 行为 padding


### 2-C  模型组件替换验证
确认 `text_model` 来自 Qwen3（`hidden_size=1024`），`connector` 为新建的 `SmolVLMConnector`，输出维度与 Qwen3 对齐。

In [ ]:
from transformers.models.smolvlm.modeling_smolvlm import SmolVLMConnector

# text_model 的嵌入维度应为 Qwen3-0.6B 的 1024
text_hidden = model.model.text_model.embed_tokens.weight.shape[1]
check(text_hidden == 1024,
      'text_model hidden_size = 1024（来自 Qwen3-0.6B）',
      f'实际={text_hidden}')

# connector 类型
check(isinstance(model.model.connector, SmolVLMConnector),
      'connector 是 SmolVLMConnector 实例',
      f'实际类型={type(model.model.connector).__name__}')

# connector 输出维度应与 Qwen3 hidden_size 一致（1024）
conn_out = model.model.connector.proj.weight.shape[0]
check(conn_out == 1024,
      'connector 输出维度 = 1024（与 Qwen3 hidden_size 对齐）',
      f'实际={conn_out}')

### 2-D  配置字段一致性验证
所有 `vocab_size`、`image_token_id`、`eos_token_id` 配置项应全部一致。

In [ ]:
IMAGE_TOKEN_ID = 151655
EOS_TOKEN_ID   = 151645

config_checks = {
    'model.vocab_size':                    (NEW_VOCAB,       model.vocab_size),
    'model.model.vocab_size':              (NEW_VOCAB,       model.model.vocab_size),
    'model.config.vocab_size':             (NEW_VOCAB,       model.config.vocab_size),
    'model.config.text_config.vocab_size': (NEW_VOCAB,       model.config.text_config.vocab_size),
    'model.image_token_id':                (IMAGE_TOKEN_ID,  model.image_token_id),
    'model.config.image_token_id':         (IMAGE_TOKEN_ID,  model.config.image_token_id),
    'generation_config.eos_token_id':      (EOS_TOKEN_ID,    model.generation_config.eos_token_id),
}

for attr, (expected, actual) in config_checks.items():
    check(actual == expected, attr, f'实际={actual}, 期望={expected}')

### 2-E  数据类型验证（bfloat16）

In [ ]:
dtype_checks = {
    'embed_tokens':   model.model.text_model.embed_tokens.weight,
    'lm_head':        model.lm_head.weight,
    'connector.proj': model.model.connector.proj.weight,
}

for name, param in dtype_checks.items():
    check(param.dtype == torch.bfloat16,
          f'{name} dtype = bfloat16',
          f'实际={param.dtype}')

### 2-F  新增位置token的嵌入向量已初始化（非零）
`resize_token_embeddings` 使用均值初始化，新增行的 L2 范数应大于 0。

In [ ]:
embed_weight  = model.model.text_model.embed_tokens.weight  # [vocab, 1024]
new_token_ids = tokenizer.convert_tokens_to_ids(ROW_COL_TOKENS)

new_embeds = embed_weight[new_token_ids].float()  # [36, 1024]
norms = new_embeds.norm(dim=-1)                    # [36]

print(f'新增token嵌入向量 L2 范数统计:')
print(f'  min={norms.min():.4f},  max={norms.max():.4f},  mean={norms.mean():.4f}')

check((norms > 0).all().item(),
      '所有新增位置token的嵌入向量均已初始化（L2范数 > 0）')

### 2-G  虚拟前向传播（纯文本）
验证模型可以正常前向，输出的 logits 形状和数值均正常。

In [ ]:
model.eval()

dummy_text = '你好，世界！'
inputs = tokenizer(dummy_text, return_tensors='pt').to(DEVICE)
seq_len = inputs['input_ids'].shape[1]
print(f'输入序列长度: {seq_len}')

# logits 的最后一维 = lm_head 输出维度 = QWEN3_MODEL_VOCAB（151936）
expected_vocab_dim = model.lm_head.weight.shape[0]

with torch.no_grad():
    try:
        outputs = model(**inputs, pixel_values=None)
        logits = outputs.logits
        print(f'输出 logits 形状: {logits.shape}  (期望: [1, {seq_len}, {expected_vocab_dim}])')
        check(logits.shape == (1, seq_len, expected_vocab_dim),
              '前向传播 logits 形状正确',
              f'实际={tuple(logits.shape)}')
        check(not logits.isnan().any().item(),
              '前向传播输出无 NaN')
        check(not logits.isinf().any().item(),
              '前向传播输出无 Inf')
    except Exception as e:
        print(f'⚠️  前向传播异常（可能需要图像输入）: {e}')

---
## 汇总报告

In [ ]:
print('=' * 60)
print('所有测试通过 ✅')
print('=' * 60)
print()
print('load_processor 验证项目:')
print('  1-A  36个 <row_i_col_j> 均为单一原子token（无BPE拆分）')
print(f'  1-B  词表大小正确（{QWEN3_BASE_VOCAB} + 36 = {QWEN3_BASE_VOCAB + 36}）')
print(f'       所有位置token的ID在扩充区域（>= {QWEN3_BASE_VOCAB}）')
print('  1-C  所有特殊token配置正确')
print('  1-D  图像处理器 size=1536, do_image_splitting=True, 网格≤6×6')
print('  1-E  聊天模板非空')
print('  1-F  端到端分词保持原子性')
print()
print('load_model_v2 验证项目:')
print(f'  2-A  不传 new_vocab_size 时 embed/lm_head 大小一致且未 resize'
      f'（模型嵌入={QWEN3_MODEL_VOCAB}，tokenizer 词表={QWEN3_BASE_VOCAB}，差值为对齐填充）')
print('  2-B  传入 new_vocab_size 后 embed_tokens 和 lm_head 均已扩充')
print('  2-C  text_model 来自 Qwen3（hidden_size=1024）')
print('       connector 已替换为新 SmolVLMConnector，输出维度=1024')
print('  2-D  所有配置字段（vocab_size / image_token_id / eos_token_id）一致')
print('  2-E  所有主要参数为 bfloat16')
print('  2-F  新增位置token的嵌入向量已初始化（L2范数 > 0）')
print('  2-G  前向传播 logits 形状正确且无 NaN/Inf')